In [7]:
import notebook_setup

In [8]:
import pandas as pd

from notebook_helpers import get_db

from app.models.anime import Anime
from app.models.genre import Genre
from app.models.anime_genre import AnimeGenre

In [9]:
db = get_db()

In [10]:
anime_df = pd.read_csv("../dataset/clean_anime.csv")

genre_df = pd.read_csv("../dataset/clean_genres.csv")

In [11]:
print(anime_df.shape)

print(genre_df.shape)

(12294, 7)
(44, 1)


In [12]:
genre_objects = []

for index, row in genre_df.iterrows():

    genre_objects.append(
        Genre(
            genre_name=row["genre_name"]
        )
    )

db.bulk_save_objects(genre_objects)

db.commit()

print("Genres Imported Successfully!")

Genres Imported Successfully!


In [13]:
genre_map = {}

genres = db.query(Genre).all()

for genre in genres:

    genre_map[genre.genre_name] = genre.genre_id

print(genre_map)

{'Action': 1, 'Adventure': 2, 'Cars': 3, 'Comedy': 4, 'Dementia': 5, 'Demons': 6, 'Drama': 7, 'Ecchi': 8, 'Fantasy': 9, 'Game': 10, 'Harem': 11, 'Hentai': 12, 'Historical': 13, 'Horror': 14, 'Josei': 15, 'Kids': 16, 'Magic': 17, 'Martial Arts': 18, 'Mecha': 19, 'Military': 20, 'Music': 21, 'Mystery': 22, 'Parody': 23, 'Police': 24, 'Psychological': 25, 'Romance': 26, 'Samurai': 27, 'School': 28, 'Sci-Fi': 29, 'Seinen': 30, 'Shoujo': 31, 'Shoujo Ai': 32, 'Shounen': 33, 'Shounen Ai': 34, 'Slice of Life': 35, 'Space': 36, 'Sports': 37, 'Super Power': 38, 'Supernatural': 39, 'Thriller': 40, 'Unknown': 41, 'Vampire': 42, 'Yaoi': 43, 'Yuri': 44}


In [14]:
anime_objects = []

for _, row in anime_df.iterrows():

    anime_objects.append(

        Anime(

            anime_id=int(row["anime_id"]),

            title=row["name"],

            type=row["type"],

            episodes=int(row["episodes"]),

            rating=float(row["rating"]),

            members=int(row["members"])

        )

    )

db.bulk_save_objects(anime_objects)

db.commit()

print("Anime Imported Successfully!")

Anime Imported Successfully!


In [20]:
# If the previous insert failed
db.rollback()

anime_genre_objects = []
seen_pairs = set()

for _, row in anime_df.iterrows():

    genres = {
        genre.strip()
        for genre in row["genre"].split(",")
    }

    for genre in genres:

        pair = (int(row["anime_id"]), genre_map[genre])

        if pair not in seen_pairs:

            seen_pairs.add(pair)

            anime_genre_objects.append(
                AnimeGenre(
                    anime_id=pair[0],
                    genre_id=pair[1]
                )
            )
print("Total relationship objects:", len(anime_genre_objects))
print("Total unique pairs:", len(seen_pairs))
db.bulk_save_objects(anime_genre_objects)
db.commit()

print("Anime-Genre Relationships Imported!")

Total relationship objects: 36346
Total unique pairs: 36346
Anime-Genre Relationships Imported!


In [17]:
db.close()
db = get_db()

In [21]:
print("Anime:", db.query(Anime).count())
print("Genres:", db.query(Genre).count())
print("Relationships:", db.query(AnimeGenre).count())

Anime: 12294
Genres: 44
Relationships: 36346


In [22]:
sorted(genre_map.keys())

['Action',
 'Adventure',
 'Cars',
 'Comedy',
 'Dementia',
 'Demons',
 'Drama',
 'Ecchi',
 'Fantasy',
 'Game',
 'Harem',
 'Hentai',
 'Historical',
 'Horror',
 'Josei',
 'Kids',
 'Magic',
 'Martial Arts',
 'Mecha',
 'Military',
 'Music',
 'Mystery',
 'Parody',
 'Police',
 'Psychological',
 'Romance',
 'Samurai',
 'School',
 'Sci-Fi',
 'Seinen',
 'Shoujo',
 'Shoujo Ai',
 'Shounen',
 'Shounen Ai',
 'Slice of Life',
 'Space',
 'Sports',
 'Super Power',
 'Supernatural',
 'Thriller',
 'Unknown',
 'Vampire',
 'Yaoi',
 'Yuri']